# Part 5 · 01 Raw cache

The only notebook in part 5 that talks to EDGAR. It downloads the chosen document of every event once more and keeps its **raw bytes**, undecoded, in `cache/raw/raw_YYYY.zip`, with one line per fetch in `cache/raw_manifest.parquet`. `fetch_releases.py` kept only its cleaned text, so the HTML has to be fetched again — but only this once: cleaning then runs on the cache and never returns to EDGAR.

**Which document:** the one `fetch_releases.py` chose. `files csv/extraction_report.csv` names it for each filing (`accession`, `cik`, `filename`), so the script goes straight to it, one request per document. Only if no such report is found does it read each index page again and apply `fetch_releases.py`'s own rules (`parse_index`, `to_text`, `looks_like_prose`). The 943 filings where `fetch_releases.py` found no prose document are not fetched.

**Rerun-safe:** archives are committed every 500 events; after an interruption (kernel restart, lost connection, machine asleep), rerun the setup cell and the same pull cell, and only the missing events are fetched. The SEC allows 10 requests a second with a descriptive User-Agent; the script uses 9.

**Before running:** run `00_setup_check` once.

In [ ]:
%pip install -q pandas numpy pyarrow beautifulsoup4 lxml nltk datasketch tiktoken transformers
# %pip installs into the environment of this kernel. If it upgraded a package that
# was already loaded, restart the kernel and continue from the next cell.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Find the checkout from the repository root or any notebook subfolder.
_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts").is_dir()
                  and (p / "3_4_surprise_returns" / "code").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the cloned S08 repository, "
                            "or set S08_ROOT to its path.")
# Code always comes from this checkout. Private data (cache/, data/) live under
# TEAM_DIR: the checkout itself unless TEAM_DIR is set.
TEAM_DIR = Path(os.environ.get("TEAM_DIR", REPO_ROOT)).expanduser().resolve()
STEP5_DIR = Path(os.environ.get("STEP5_DIR", TEAM_DIR / "5_data cleaning")).expanduser().resolve()
CODE = REPO_ROOT / "5_data cleaning" / "code"

for sub in ["cache", "data", "results", "logs"]:
    (STEP5_DIR / sub).mkdir(parents=True, exist_ok=True)

# settings.py, and every script started below, read the roots from these.
os.environ["S08_ROOT"] = str(REPO_ROOT)   # rerunning this cell after the chdir still finds it
os.environ["TEAM_DIR"] = str(TEAM_DIR)
os.environ["STEP5_DIR"] = str(STEP5_DIR)
# CODE goes first on the path and part 5's modules are read again from it: a kernel
# that already imported an older settings.py (or part 7-8's) would keep serving it.
while str(CODE) in sys.path:
    sys.path.remove(str(CODE))
sys.path.insert(0, str(CODE))
for _m in ["settings", "cache_raw", "clean_release", "test_clean_release", "entities",
           "token_stats", "dedup", "model_cache", "splits", "manifest"]:
    sys.modules.pop(_m, None)
os.chdir(STEP5_DIR)       # relative artifacts stay in the private data folder


def run_script(name, *args):
    """Run code/<name> with this kernel's Python, show its output here as it comes,
    and stop the notebook if it fails."""
    cmd = [sys.executable, "-u", str(CODE / name), *map(str, args)]
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                          text=True, encoding="utf-8", errors="replace") as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)

missing = [f for f in ["settings.py", "cache_raw.py"] if not (CODE / f).exists()]
assert not missing, f"Missing code in {CODE}: {missing}"
print("repository:   ", REPO_ROOT)
print("data root:    ", TEAM_DIR)
print("step 5 folder:", STEP5_DIR)
print("code files:   ", sorted(os.listdir(CODE)))

## Test on 200 events

One to two minutes, with a progress line every 10 events (events, documents found, failures, speed, time left). Then check that the documents are the ones `fetch_releases.py` chose before the full pass.

In [ ]:
# run_script("cache_raw.py", "--limit", 200)

In [ ]:
import pandas as pd
import settings as S

man = pd.read_parquet(S.RAW_MANIFEST)
docs = man[man.kind == "doc"]
rep = pd.read_csv(S.EXTRACTION_REPORT, dtype=str, keep_default_na=False).drop_duplicates("accession")
m = docs.merge(rep[["accession", "source", "filename"]], on="accession", how="left",
               suffixes=("", "_report"))
failed = docs.source.isin(["fetch failed", "index fetch failed"])
print(f"events fetched: {len(docs):,} | documents stored: {int((docs.sha256 != '').sum()):,} | "
      f"failed, tried again on the next run: {int(failed.sum()):,}")
print(docs.source.value_counts().to_string())
print(f"\nsame document as fetch_releases.py chose: {(m.filename == m.filename_report)[~failed.values].mean():.1%}")

## Full pass

About 117,000 documents: 3 to 4 hours with the extraction report (7 to 8 in index mode).


A download that fails for a passing reason (no answer, SEC refusal, server error) is marked `fetch failed`, never `none`, and is tried again whenever this cell is rerun. At the end, rerun the cell until the `failed` count is 0 or stops falling; the check cell above shows what is left.

In [ ]:
RUN_FULL = False

if RUN_FULL:
    run_script("cache_raw.py")
else:
    print("full pass not run: set RUN_FULL = True")